# LULC / Vegetation Source Inputs — Visual QA

Inspect the three raw source rasters in `input/lulc_veg/`:

| File | Variable | Native dtype | NoData |
|------|----------|-------------|--------|
| `CNPY.tif` | Tree canopy cover | int16 | −32 768 |
| `Imperv.tif` | Impervious surface fraction | uint8 | 255 |
| `RootDepth.tif` | Root zone depth | int8 | −128 |

All three are CONUS-scale 30 m rasters in **ESRI:102039** (USA Contiguous Albers Equal Area Conic — USGS).  
Reads use rasterio's `out_shape` decimated read so only a thumbnail is decompressed.

> **Note on `RootDepth.tif` zeros:** Value 0 is **not** the file's NoData (−128 is).  
> Downstream, `resample()` explicitly converts 0 → NaN via `mask_values=(128, 0)`,  
> treating annual cropland (which carries no natural root‑depth estimate) as missing.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.enums import Resampling

DATA_ROOT  = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2_param_v2")
LULC_DIR   = DATA_ROOT / "input" / "lulc_veg"

# Target display width in pixels — all reads are decimated to this
DISPLAY_PX = 2000


## Available files

In [ ]:
for p in sorted(LULC_DIR.glob("*.tif")):
    ok = False
    try:
        with rasterio.open(p):
            ok = True
    except Exception:
        pass
    status  = "\u2713" if ok else "\u2717 (corrupt/partial)"
    size_mb = round(p.stat().st_size / 1024**2)
    print(f"  {status}  {size_mb:>6} MB  {p.name}")


## Helper functions

In [ ]:

def raster_meta(path: Path) -> dict:
    """Return metadata for a raster without loading pixel data."""
    with rasterio.open(path) as src:
        return {
            "shape":    src.shape,
            "dtype":    src.dtypes[0],
            "crs":      src.crs,
            "bounds":   src.bounds,
            "res_m":    abs(src.transform.a),
            "nodata":   src.nodata,
            "overviews": src.overviews(1),
            "size_MB":  round(path.stat().st_size / 1024**2),
        }


def decimated_read(path: Path, target_px: int = DISPLAY_PX,
                   resampling=Resampling.average,
                   extra_mask_values: tuple = ()):
    """
    Read a decimated thumbnail via rasterio out_shape.

    extra_mask_values: additional pixel values to treat as NoData
    (e.g. pass extra_mask_values=(0,) for RootDepth to preview the
     same masking that resample() applies downstream).
    """
    with rasterio.open(path) as src:
        factor = max(1, max(src.width, src.height) // target_px)
        out_h  = max(1, src.height // factor)
        out_w  = max(1, src.width  // factor)
        data   = src.read(
            1,
            out_shape=(out_h, out_w),
            resampling=resampling,
        ).astype(np.float64)
        nodata = src.nodata
        t = src.transform
        scaled_transform = t * t.scale(src.width / out_w, src.height / out_h)
        meta = raster_meta(path)

    mask = ~np.isfinite(data)
    if nodata is not None:
        mask |= data == nodata
    for val in extra_mask_values:
        mask |= data == val
    return np.ma.array(data, mask=mask), scaled_transform, meta


def percentile_stretch(arr, lo=2, hi=98):
    valid = arr.compressed() if isinstance(arr, np.ma.MaskedArray) else arr[np.isfinite(arr)]
    if valid.size == 0:
        return arr, 0.0, 1.0
    vmin, vmax = np.percentile(valid, [lo, hi])
    return np.clip(arr, vmin, vmax), vmin, vmax


def plot_raster(path: Path, title: str, cmap: str = "viridis", units: str = "",
                stretch_pct: tuple = (2, 98),
                extra_mask_values: tuple = ()):
    """Display a large raster as a decimated overview with a distribution histogram."""
    if not path.exists():
        print(f"  Skipping {path.name} — not found")
        return
    try:
        data, _, meta = decimated_read(path, extra_mask_values=extra_mask_values)
    except Exception as e:
        print(f"  Cannot read {path.name}: {e}")
        return

    stretched, vmin, vmax = percentile_stretch(data, lo=stretch_pct[0], hi=stretch_pct[1])
    valid = data.compressed()

    if valid.size == 0:
        print(f"  {path.name} — thumbnail entirely masked (check nodata settings).")
        plt.close()
        return

    p25, p50, p75 = np.percentile(valid, [25, 50, 75])
    h, w = meta["shape"]

    fig, (ax_img, ax_hist) = plt.subplots(
        1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [3, 1]}
    )

    im = ax_img.imshow(stretched, cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
    ax_img.set_title(f"{title}\n{path.name}  ({meta['size_MB']} MB)", fontsize=10)
    ax_img.axis("off")
    plt.colorbar(im, ax=ax_img, fraction=0.03, pad=0.02, label=units)

    ax_hist.hist(valid, bins=150, color="steelblue", edgecolor="none", density=True)
    ax_hist.axvline(p25,  color="cornflowerblue", linestyle="--", linewidth=1.2, label=f"p25 = {p25:.3g}")
    ax_hist.axvline(p50,  color="navy",           linestyle="-",  linewidth=1.8, label=f"p50 = {p50:.3g}")
    ax_hist.axvline(p75,  color="cornflowerblue", linestyle="--", linewidth=1.2, label=f"p75 = {p75:.3g}")
    ax_hist.axvline(vmin, color="orange",         linestyle=":",  linewidth=1.2, label=f"stretch lo = {vmin:.3g}")
    ax_hist.axvline(vmax, color="red",            linestyle=":",  linewidth=1.2, label=f"stretch hi = {vmax:.3g}")
    ax_hist.set_title(
        f"{h:,} \u00d7 {w:,} px  |  {meta['res_m']:.0f} m\n"
        f"valid: {valid.size:,}  mean: {valid.mean():.4g}  std: {valid.std():.4g}",
        fontsize=8, loc="left", family="monospace",
    )
    ax_hist.set_xlabel(units or "value")
    ax_hist.set_ylabel("density")
    ax_hist.legend(fontsize=8)

    plt.tight_layout()
    plt.show()


## Tree Canopy Cover (`CNPY.tif`)

National-scale percent tree canopy cover at 30 m, ESRI:102039.

- **Dtype / NoData:** int16 / −32 768
- **Units:** % canopy cover (0–100)
- **Expected range:** 0 % over open water, urban, and cropland; 60–95 % over dense forest
- **Distribution:** Strongly bimodal — large mass near 0 (non-forest CONUS majority), secondary peak around 60–90 % for forested regions
- **Downstream use:** Resampled to the LULC grid → `cnpy_resampled_lulc.tif`; combined with winter retention to derive radiation transmission (`radtrn`)

In [ ]:
plot_raster(LULC_DIR / "CNPY.tif", "Tree Canopy Cover", cmap="Greens", units="%")


## Impervious Surface (`Imperv.tif`)

Percent impervious surface cover at 30 m, ESRI:102039.

- **Dtype / NoData:** uint8 / 255
- **Units:** % impervious cover (0–100)
- **Expected range:** 0 % for natural/vegetated land; 80–100 % for dense urban cores; intermediate values in suburban fringe
- **Distribution:** Highly right-skewed / zero-inflated — the vast majority of CONUS pixels are 0 (natural land cover); the upper tail extends to 100 % in cities
- **Downstream use:** Not directly used in `soil_moist_max` or `radtrn`, but informs HRU impervious fraction parameters in NHM

> **Hint:** The colormap stretch will compress most of the CONUS to the low end. Try `stretch_pct=(50, 100)` to highlight the urban signal.

In [ ]:
plot_raster(LULC_DIR / "Imperv.tif", "Impervious Surface Cover", cmap="hot_r", units="%")


### Urban signal only — stretch p50–p100

Re-plot with a tighter stretch to reveal spatial patterns in the non-zero urban/suburban pixels.

In [ ]:
plot_raster(LULC_DIR / "Imperv.tif", "Impervious Surface Cover (urban signal)",
            cmap="hot_r", units="%", stretch_pct=(50, 100))


## Root Zone Depth — raw file (`RootDepth.tif`)

Effective root zone depth at 30 m (native 250 m resolution, stored here as 30 m), ESRI:102039.

- **Dtype / NoData:** int8 / −128 (equivalent to unsigned uint8 value 128, which `resample()` also masks)
- **Units:** cm
- **Expected range:** 0–127 cm; value **0 encodes annual cropland** (no natural root-depth estimate assigned by the source product)
- **Distribution:** Bimodal with a large peak at 0 (crops/bare) and a secondary peak for natural vegetation classes
- **⚠️ White areas in plots:** The Great Plains, CA Central Valley, and Midwest appear white because those pixels carry value 0, which `resample()` converts to NaN downstream via `mask_values=(128, 0)`

Two versions are plotted:
1. **As stored** — 0 kept as a valid value (file NoData = −128 only)
2. **As used downstream** — 0 also masked, matching `resample()` behaviour

In [ ]:
# As stored in the file — zero pixels (cropland) visible
plot_raster(LULC_DIR / "RootDepth.tif", "Root Zone Depth (crops shown as 0 cm)",
            cmap="YlOrBr", units="cm")


In [ ]:
# As used downstream — zero pixels masked out to match resample() behaviour
plot_raster(LULC_DIR / "RootDepth.tif",
            "Root Zone Depth (crops masked, matching resample() behaviour)",
            cmap="YlOrBr", units="cm",
            extra_mask_values=(0,))


## Side-by-side overview: all three inputs

In [ ]:
targets = [
    ("CNPY.tif",     "Canopy Cover (%)",       "Greens",  {},              "%"),
    ("Imperv.tif",   "Impervious (%)",          "hot_r",   {},              "%"),
    ("RootDepth.tif","Root Depth (cm)\n[crops masked]", "YlOrBr",
     {"extra_mask_values": (0,)},                                            "cm"),
]

available = [(fn, lbl, cm, kw, u) for fn, lbl, cm, kw, u in targets
             if (LULC_DIR / fn).exists()]

if not available:
    print("No files found.")
else:
    fig, axes = plt.subplots(1, len(available), figsize=(7 * len(available), 6))
    if len(available) == 1:
        axes = [axes]
    for ax, (fn, label, cmap, kw, units) in zip(axes, available):
        try:
            data, _, _ = decimated_read(LULC_DIR / fn, **kw)
            stretched, vmin, vmax = percentile_stretch(data)
            im = ax.imshow(stretched, cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
            ax.set_title(label, fontsize=10)
            ax.axis("off")
            plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label=units)
        except Exception as e:
            ax.set_title(f"{label}\n(error: {e})")
            ax.axis("off")
    plt.suptitle("lulc_veg source inputs — decimated overview (p2–p98 stretch)", fontsize=12)
    plt.tight_layout()
    plt.show()
